# Análise de Transações

## Imports e constantes

In [183]:
import csv
import json
from datetime import datetime
from pathlib import Path

LIMITE_SUSPEITO = 10000.00
CSV_FILE_PATH = 'transacoes.csv'

## Funções

In [184]:
def ler_transacoes(caminho=CSV_FILE_PATH):
    try:
        with Path(caminho).open(newline="", encoding="utf-8") as arquivo:
            return list(csv.DictReader(arquivo))
    except FileNotFoundError:
        print(f"Arquivo nao encontrado: {caminho}")
        return []

In [185]:
def validar_transacao(linha):
    try:
        id_texto = linha["id"].strip()
        data_texto = linha["data"].strip()
        cliente_id = linha["cliente_id"].strip()
        tipo = linha["tipo"].strip().lower()
        valor_texto = linha["valor"].strip()
    except KeyError:
        return None

    if not id_texto.isdigit() or not cliente_id or tipo not in {"credito", "debito"}:
        return None

    try:
        data = datetime.strptime(data_texto, "%Y-%m-%d")
    except ValueError:
        return None

    if data.strftime("%Y-%m-%d") != data_texto:
        return None

    try:
        valor = float(valor_texto)
    except ValueError:
        return None

    if valor <= 0:
        return None

    return {
        "id": int(id_texto),
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": round(valor, 2),
        "descricao": linha.get("descricao", "").strip(),
        "categoria": linha.get("categoria", "").strip(),
    }

In [186]:
def exibir_resumo_limpeza(estatisticas):
    print("===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {estatisticas['total_linhas']}")
    print(f"Linhas validas: {estatisticas['linhas_validas']}")
    print(f"Linhas invalidas: {estatisticas['linhas_invalidas']}")
    print()

In [187]:
def processar_transacoes(linhas):
    transacoes_validas = []
    ids_usados = set()
    total_linhas = 0
    linhas_invalidas = 0

    for linha in linhas:
        total_linhas += 1
        transacao = validar_transacao(linha)

        if transacao is None or transacao["id"] in ids_usados:
            linhas_invalidas += 1
            continue

        ids_usados.add(transacao["id"])
        transacoes_validas.append(transacao)

    estatisticas = {
        "total_linhas": total_linhas,
        "linhas_validas": len(transacoes_validas),
        "linhas_invalidas": linhas_invalidas,
    }
    return transacoes_validas, estatisticas


linhas = ler_transacoes()
transacoes, estatisticas = processar_transacoes(linhas)
exibir_resumo_limpeza(estatisticas)

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 20
Linhas validas: 15
Linhas invalidas: 5



In [188]:
def gerar_relatorio(transacoes, estatisticas):
    resumo_mensal = {}
    suspeitas = []

    for transacao in transacoes:
        mes = transacao["data"].strftime("%Y-%m")
        valor = transacao["valor"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": valor,
                "menor_valor": valor,
                "_soma_valores": 0.0,
            }

        resumo = resumo_mensal[mes]
        resumo["quantidade"] += 1
        resumo["_soma_valores"] += valor

        if transacao["tipo"] == "credito":
            resumo["total_credito"] += valor
        else:
            resumo["total_debito"] += valor

        resumo["maior_valor"] = max(resumo["maior_valor"], valor)
        resumo["menor_valor"] = min(resumo["menor_valor"], valor)

        if valor > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data"].strftime("%Y-%m-%d"),
                "valor": valor,
            })

    resumo_final = {}
    for mes in sorted(resumo_mensal):
        resumo = resumo_mensal[mes]
        resumo["total_credito"] = round(resumo["total_credito"], 2)
        resumo["total_debito"] = round(resumo["total_debito"], 2)
        resumo["saldo"] = round(resumo["total_credito"] - resumo["total_debito"], 2)
        resumo["media"] = round(resumo["_soma_valores"] / resumo["quantidade"], 2)
        del resumo["_soma_valores"]
        resumo_final[mes] = resumo

    if transacoes:
        datas = [transacao["data"] for transacao in transacoes]
        data_inicial = min(datas)
        data_final = max(datas)
        periodo = {"inicio": data_inicial.strftime("%Y-%m-%d"), "fim": data_final.strftime("%Y-%m-%d")}
        dias_entre = (data_final - data_inicial).days
    else:
        periodo = {"inicio": None, "fim": None}
        dias_entre = 0

    dados_relatorio = {
        "gerado_em": datetime.now().date().isoformat(),
        "total_transacoes_validas": estatisticas["linhas_validas"],
        "total_transacoes_invalidas": estatisticas["linhas_invalidas"],
        "resumo_mensal": resumo_final,
        "transacoes_suspeitas": suspeitas,
    }

    info_extra = {
        "periodo": periodo,
        "dias_entre_transacoes": dias_entre,
    }

    return dados_relatorio, info_extra


relatorio, info_extra = gerar_relatorio(transacoes, estatisticas)

print(f"Meses analisados: {', '.join(relatorio['resumo_mensal'].keys())}")
print(f"Periodo analisado: {info_extra['periodo']['inicio']} -> {info_extra['periodo']['fim']}")
print(f"Dias entre transacoes: {info_extra['dias_entre_transacoes']}")
print(f"Transacoes suspeitas: {len(relatorio['transacoes_suspeitas'])}")

Meses analisados: 2026-01, 2026-02, 2026-03, 2026-04
Periodo analisado: 2026-01-05 -> 2026-04-20
Dias entre transacoes: 105
Transacoes suspeitas: 2


In [189]:
def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

In [190]:
def exibir_relatorio(relatorio):
    print("===== RELATORIO MENSAL =====")
    print()

    for mes, resumo in relatorio["resumo_mensal"].items():
        print(f"Mes: {mes}")
        print(f"Transacoes: {resumo['quantidade']}")
        print(f"Total credito: {formatar_moeda(resumo['total_credito'])}")
        print(f"Total debito:  {formatar_moeda(resumo['total_debito'])}")
        print(f"Saldo:         {formatar_moeda(resumo['saldo'])}")
        print(f"Media:         {formatar_moeda(resumo['media'])}")
        print(f"Maior valor:   {formatar_moeda(resumo['maior_valor'])}")
        print(f"Menor valor:   {formatar_moeda(resumo['menor_valor'])}")
        print("-" * 32)

    print()
    print("===== TRANSACOES SUSPEITAS =====")
    if not relatorio["transacoes_suspeitas"]:
        print("Nenhuma transacao suspeita encontrada.")
        return

    for transacao in relatorio["transacoes_suspeitas"]:
        print(
            "ID: {id} | Cliente: {cliente_id} | Data: {data} | Valor: {valor}".format(
                id=transacao["id"],
                cliente_id=transacao["cliente_id"],
                data=transacao["data"],
                valor=formatar_moeda(transacao["valor"]),
            )
        )

In [191]:
def salvar_json(relatorio, caminho="relatorio.json"):
    with Path(caminho).open("w", encoding="utf-8") as arquivo:
        json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)

## Script Principal

In [192]:
if __name__ == "__main__":
  linhas = ler_transacoes()
  transacoes, estatisticas = processar_transacoes(linhas)

  relatorio = gerar_relatorio(transacoes, estatisticas)[0]

  exibir_relatorio(relatorio)

  caminho_json = "relatorio.json"
  salvar_json(relatorio, caminho_json)

  print()
  print(f"Relatorio salvo em: {caminho_json}")

===== RELATORIO MENSAL =====

Mes: 2026-01
Transacoes: 4
Total credito: R$ 4.700,00
Total debito:  R$ 430,50
Saldo:         R$ 4.269,50
Media:         R$ 1.282,62
Maior valor:   R$ 3.500,00
Menor valor:   R$ 180,50
--------------------------------
Mes: 2026-02
Transacoes: 4
Total credito: R$ 15.000,00
Total debito:  R$ 609,90
Saldo:         R$ 14.390,10
Media:         R$ 3.902,47
Maior valor:   R$ 15.000,00
Menor valor:   R$ 89,90
--------------------------------
Mes: 2026-03
Transacoes: 4
Total credito: R$ 25.500,00
Total debito:  R$ 739,90
Saldo:         R$ 24.760,10
Media:         R$ 6.559,98
Maior valor:   R$ 22.000,00
Menor valor:   R$ 99,90
--------------------------------
Mes: 2026-04
Transacoes: 3
Total credito: R$ 6.850,00
Total debito:  R$ 800,00
Saldo:         R$ 6.050,00
Media:         R$ 2.550,00
Maior valor:   R$ 4.100,00
Menor valor:   R$ 800,00
--------------------------------

===== TRANSACOES SUSPEITAS =====
ID: 6 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.00